# Demo Dự đoán Bệnh Vật nuôi & Trực quan hóa Grad-CAM

Notebook này cho phép thực hiện suy luận (Inference) chẩn đoán bệnh từ một ảnh đầu vào bất kỳ cho **Bò (Cow)**, **Gà (Chicken)**, hoặc **Lợn (Pig)** bằng mô hình **ResNet-18** đã fine-tune (Phase 2).

Đồng thời, hệ thống sử dụng **Grad-CAM (Gradient-weighted Class Activation Mapping)** để sinh nhiệt đồ (heatmap) trực quan hóa vùng chú ý mà mô hình tập trung quan sát khi đưa ra quyết định.

In [ ]:
import sys
from pathlib import Path
import torch
from PIL import Image
import matplotlib.pyplot as plt

# 1. Xác định thư mục gốc của project (PROJECT_ROOT)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.load_model import load_best_model
from src.gradcam import predict_with_gradcam
from src.utils import ANIMAL_CLASSES, ANIMAL_DISPLAY_NAMES, get_pipeline_output_paths

# -------------------------------------------------------------
# CẤU HÌNH SUY LUẬN (INFERENCE CONFIG)
# -------------------------------------------------------------
# 1. Chọn loài vật cần chẩn đoán: "cow" (Bò), "chicken" (Gà), hoặc "pig" (Lợn)
ANIMAL = "chicken"

# 2. Đường dẫn đến file ảnh cần dự đoán (định dạng JPG hoặc PNG)
CUSTOM_IMAGE_PATH = PROJECT_ROOT / "figures" / "sample_test.jpg"

# 3. Tự động xác định đường dẫn checkpoint và danh sách nhãn phân loại
paths = get_pipeline_output_paths(PROJECT_ROOT, animal=ANIMAL, phase="phase2")
MODEL_PATH = paths["model_checkpoint"]
CLASS_NAMES = ANIMAL_CLASSES[ANIMAL]
DISPLAY_NAMES = ANIMAL_DISPLAY_NAMES.get(ANIMAL, CLASS_NAMES)

# 4. Xác định thiết bị tính toán (GPU nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 65)
print(f"HỆ THỐNG SUY LUẬN VÀ TRỰC QUAN HÓA GRAD-CAM: {ANIMAL.upper()}")
print("=" * 65)
print(f"Trọng số mô hình (Checkpoint): {MODEL_PATH.name}")
print(f"Số lượng lớp chẩn đoán       : {len(CLASS_NAMES)}")
print(f"Thiết bị tính toán (Device)  : {device}")
print(f"Đường dẫn ảnh kiểm tra       : {CUSTOM_IMAGE_PATH}")
print("=" * 65)

In [ ]:
# 1. Nạp mô hình ResNet-18 đã được fine-tune tốt nhất
model = load_best_model(MODEL_PATH, n_classes=len(CLASS_NAMES), device=device)

# 2. Kiểm tra file ảnh đầu vào
if not Path(CUSTOM_IMAGE_PATH).exists():
    print(f"\n[CHÚ Ý] File ảnh mẫu chưa tồn tại tại: {CUSTOM_IMAGE_PATH}")
    print("Vui lòng thay đổi biến `CUSTOM_IMAGE_PATH` trỏ tới ảnh thực tế trên máy của bạn để chạy dự đoán.")
else:
    # 3. Mở ảnh và thực thi dự đoán kèm Grad-CAM
    image = Image.open(CUSTOM_IMAGE_PATH)
    result = predict_with_gradcam(model, image, device, class_names=CLASS_NAMES)

    # 4. In kết quả chẩn đoán chi tiết ra màn hình
    print("\n" + "=" * 50)
    print("              KẾT QUẢ CHẨN ĐOÁN")
    print("=" * 50)
    print(f"Chẩn đoán dự đoán : {result['predicted_class']}")
    print(f"Độ tin cậy        : {result['confidence_percentage']:.2f}%")
    print("-" * 50)
    print("Top dự đoán hàng đầu:")
    for item in result["top5"]:
        print(f"  • {item['class_name']:<30}: {item['percentage']:.2f}%")
    print("=" * 50)

    # 5. Trực quan hóa ảnh gốc và ảnh nhiệt Grad-CAM cạnh nhau
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(result["rgb_image_np"])
    axes[0].set_title("Ảnh đầu vào (Đã tiền xử lý)", fontsize=11, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(result["gradcam_image_np"])
    axes[1].set_title(
        f"Nhiệt đồ Grad-CAM\n({result['predicted_class']}: {result['confidence_percentage']:.1f}%)",
        fontsize=11,
        fontweight="bold",
        color="darkblue",
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()